In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Load data from your CSVs

def load_yfinance_csv(path, asset_name):
    df = pd.read_csv(path)
    df = df.iloc[2:].copy()
    df.rename(columns={"Price": "Date"}, inplace=True)

    df = df[["Date", "Close"]].copy()
    df["Date"] = pd.to_datetime(df["Date"])

    df[asset_name] = pd.to_numeric(df["Close"], errors="coerce")
    df = df[["Date", asset_name]].dropna()

    return df


# Load CVX and XOM
cvx = load_yfinance_csv("CVX_daily.csv", "CVX")
xom = load_yfinance_csv("XOM_daily.csv", "XOM")

df = pd.merge(cvx, xom, on="Date", how="inner")
df = df.sort_values("Date").reset_index(drop=True)

df

# Compute returns and log prices

df["CVX"] = pd.to_numeric(df["CVX"], errors="coerce")
df["XOM"] = pd.to_numeric(df["XOM"], errors="coerce")

df = df.dropna(subset=["CVX", "XOM"])
df = df[(df["CVX"] > 0) & (df["XOM"] > 0)].copy()
df = df.sort_values("Date").reset_index(drop=True)

df["CVX_ret"] = df["CVX"].pct_change()
df["XOM_ret"] = df["XOM"].pct_change()

df["log_CVX"] = np.log(df["CVX"])
df["log_XOM"] = np.log(df["XOM"])

# Rolling beta, spread, and z-score

lookback = 252
entry_z = 1.6
exit_z = 0.5

df["beta"] = np.nan
df["spread"] = np.nan
df["zscore"] = np.nan

for i in range(lookback, len(df)):
    train = df.iloc[i - lookback:i]

    x = train["log_CVX"].values
    y = train["log_XOM"].values

    beta = np.polyfit(x, y, 1)[0]
    df.loc[i, "beta"] = beta

    train_spread = train["log_XOM"] - beta * train["log_CVX"]
    mu = train_spread.mean()
    sigma = train_spread.std()

    current_spread = df.loc[i, "log_XOM"] - beta * df.loc[i, "log_CVX"]

    df.loc[i, "spread"] = current_spread
    df.loc[i, "zscore"] = 0.0 if sigma == 0 else (current_spread - mu) / sigma

# Trading position

df["position"] = 0.0
current_pos = 0

for i in range(lookback, len(df)):
    z = df.loc[i, "zscore"]

    if current_pos == 0:
        if z > entry_z:
            current_pos = -1  # short spread: short XOM, long CVX
        elif z < -entry_z:
            current_pos = 1  # long spread: long XOM, short CVX

    elif current_pos == 1:
        if z > -exit_z:
            current_pos = 0

    elif current_pos == -1:
        if z < exit_z:
            current_pos = 0

    df.loc[i, "position"] = current_pos

# Strategy metrics

df["beta_ffill"] = df["beta"].ffill()

# Spread return follows:
# spread = log(XOM) - beta * log(CVX)
# so spread_ret approx = XOM_ret - beta * CVX_ret
df["spread_ret"] = df["XOM_ret"] - df["beta_ffill"] * df["CVX_ret"]

# Lag position so we only earn returns from positions entered previously
df["position_lag"] = df["position"].shift(1).fillna(0)

df["strategy_ret"] = df["position_lag"] * df["spread_ret"]
df["strategy_ret"] = df["strategy_ret"].fillna(0)

df["equity_curve"] = (1 + df["strategy_ret"]).cumprod()

mean_ret = df["strategy_ret"].mean()
std_ret = df["strategy_ret"].std()

sharpe = np.nan if std_ret == 0 else np.sqrt(252) * mean_ret / std_ret
total_return = df["equity_curve"].iloc[-1] - 1

rolling_max = df["equity_curve"].cummax()
drawdown = df["equity_curve"] / rolling_max - 1
max_drawdown = drawdown.min()

print("Sharpe Ratio:", sharpe)
print("Total Return:", total_return)
print("Max Drawdown:", max_drawdown)

# Plots

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

axes[0].plot(df["Date"], df["XOM"], label="XOM")
axes[0].plot(df["Date"], df["CVX"], label="CVX")
axes[0].set_title("Prices")
axes[0].legend()

axes[1].plot(df["Date"], df["spread"], label="Spread")
axes[1].set_title("Spread = log(XOM) - beta * log(CVX)")
axes[1].legend()

axes[2].plot(df["Date"], df["zscore"], label="Z-score")
axes[2].axhline(entry_z, linestyle="--", label="Entry Z")
axes[2].axhline(-entry_z, linestyle="--")
axes[2].axhline(exit_z, linestyle=":", label="Exit Z")
axes[2].axhline(-exit_z, linestyle=":")
axes[2].axhline(0, linestyle="-")
axes[2].set_title("Spread Z-score")
axes[2].legend()

axes[3].plot(df["Date"], df["equity_curve"], label="Equity Curve")
axes[3].set_title("Strategy Equity Curve")
axes[3].legend()

plt.tight_layout()
plt.show()